# Shielding CNN — Inference Notebook

Run single-layer or multilayer neutron dose inference using the trained k100s2 model.

**Primary models** (under `models/.../primary/`):
- `k100s2_v1`     — main single-layer model (k100s2 architecture, trained on TrackNet10 source)
- `transfer_mlp`  — MLP correction head frozen on top of k100s2_v1, used for multilayer layer-2

**Multilayer mode (recommended):**  
Layer 1 runs k100s2_v1 to extract amplitude attenuation (`a`-factor).  
Layer 2 runs transfer_mlp with the **original TrackNet10 source** (not the CNN layer-1 output) — keeping the model in-distribution.  
Final flux = layer-2 prediction × `a`-factor.

**Kernel:** use `../venv/bin/python` (Python 3.13, Keras 3.14.1) — base conda will fail to load the pkl files.

---
## 1 — Imports & paths

In [ ]:
import pickle, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Make sure the shielding_ml package is importable
REPO = Path('.').resolve()
sys.path.insert(0, str(REPO.parent))

from shielding_ml.data.loaders   import load_spectrum, load_phits
from shielding_ml.data.constants import FLUX_SCALE, MATERIALS, E_ALL
from shielding_ml.metrics.dose   import _build_r_all, dose_midpoint
from shielding_ml.pipelines.paths import (
    REPO_ROOT, TRACKNET10_SPECTRUM, DOSE_TABLE, DATA_REF,
)

# ── model paths ───────────────────────────────────────────────────────────────
MODEL_BASE  = REPO_ROOT / 'models' / 'k100s2-k25s2-k11-k3-d256-d64-mae-bins50-240'
PRIMARY     = MODEL_BASE / 'primary'
EXPERIMENTAL = MODEL_BASE / 'experimental'

print('Primary models:')
for p in sorted(PRIMARY.iterdir()):
    print(f'  {p.name}')
print('\nExperimental models:')
for p in sorted(EXPERIMENTAL.iterdir()):
    print(f'  {p.name}')

---
## 2 — Load models

In [ ]:
def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# k100s2 v1 — used for single-layer inference and layer-1 in multilayer
model_v1 = load_pkl(PRIMARY / 'k100s2_v1' / 'model.pkl')

# Transfer MLP head — used for layer-2 in multilayer inference
model_transfer = load_pkl(PRIMARY / 'transfer_mlp' / 'transfer_model.pkl')

print('k100s2_v1:')
model_v1.summary()
print('\ntransfer_mlp:')
model_transfer.summary()

---
## 3 — Shared helpers

In [ ]:
# ── constants ─────────────────────────────────────────────────────────────────
INPUT_SCALE = 0.001          # scales thickness and density inputs to ~[0,1]
BIN_SLICE   = slice(20, 150) # energy range used for amplitude factor (20–150 MeV)

# ── source spectrum and dose table ────────────────────────────────────────────
A_SPEC    = load_spectrum(str(TRACKNET10_SPECTRUM))  # normalised TrackNet10 beam, shape (250,)
VOID_FLUX, _ = load_phits(str(DATA_REF / 'No_Shielding.out'))  # unshielded reference flux
R_ALL     = _build_r_all(DOSE_TABLE)                # ICRP fluence-to-dose coefficients

# ── prediction helper ─────────────────────────────────────────────────────────
def predict_flux(model, source_spec, material, thickness_cm):
    """Run one CNN forward pass. Returns flux array (250,) in n/cm²/source."""
    density = MATERIALS[material]               # g/cm³ lookup
    B  = np.ones(250) * thickness_cm * INPUT_SCALE
    B1 = np.ones(250) * density      * INPUT_SCALE
    X  = np.array([source_spec, B, B1]).T.reshape(1, 250, 3)
    log_flux = model.predict(X, verbose=0).flatten()
    return 10 ** log_flux * FLUX_SCALE

def compute_dose(flux):
    """Compute effective dose rate (mrem/hr) from flux array."""
    return dose_midpoint(flux, R_ALL, excl_bin0=True)

print('Available materials:', list(MATERIALS.keys()))
print('Densities (g/cm³):', MATERIALS)

---
## 4 — Single-layer inference

Set `MATERIAL` and `THICKNESS_CM`, then run the cell below.

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
MATERIAL     = 'Concrete'   # 'Concrete', 'Steel', or 'BPE'
THICKNESS_CM = 45

# Optional: path to a PHITS .out file to compare against (set to None to skip)
PHITS_FILE = None
# PHITS_FILE = 'data/raw/phits/high_energy/Concrete_45cm_1.out'
# ─────────────────────────────────────────────────────────────────────────────

# Run inference
flux_cnn  = predict_flux(model_v1, A_SPEC, MATERIAL, THICKNESS_CM)
dose_cnn  = compute_dose(flux_cnn)

print(f'{MATERIAL}  {THICKNESS_CM} cm')
print(f'  CNN dose rate : {dose_cnn:.4e} mrem/hr')

# ── Plot ──────────────────────────────────────────────────────────────────────
MAT_COLOR = {'Concrete': '#4e79a7', 'Steel': '#e15759', 'BPE': '#59a14f'}
color = MAT_COLOR.get(MATERIAL, '#333333')

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(E_ALL, flux_cnn, color=color, lw=2, label='CNN model')

if PHITS_FILE:
    phits_flux, _ = load_phits(PHITS_FILE)
    dose_phits    = compute_dose(phits_flux)
    pct_err       = 100 * (dose_cnn - dose_phits) / dose_phits
    ax.semilogy(E_ALL, phits_flux, 'k-', lw=2, label='PHITS')
    ax.set_title(f'{MATERIAL}  {THICKNESS_CM} cm   |   dose error {pct_err:+.1f}%',
                 fontsize=12, fontweight='bold')
    print(f'  PHITS dose    : {dose_phits:.4e} mrem/hr')
    print(f'  Dose error    : {pct_err:+.1f}%')
else:
    ax.set_title(f'{MATERIAL}  {THICKNESS_CM} cm', fontsize=12, fontweight='bold')

ax.set_xlabel('Neutron energy (MeV)', fontsize=11)
ax.set_ylabel('Flux  [n / cm² / source n]', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.25)
plt.tight_layout()
plt.show()

---
## 5 — Multilayer inference  *(recommended mode)*

Layer 1: k100s2_v1 extracts the amplitude attenuation (`a`-factor).  
Layer 2: transfer_mlp runs with the original TrackNet10 source, then scaled by `a`.

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
MAT1   = 'Concrete';  THICK1 = 45   # first shield layer
MAT2   = 'Steel';     THICK2 = 33   # second shield layer

# Optional: PHITS multilayer reference (set to None to skip)
PHITS_ML_FILE = None
# PHITS_ML_FILE = 'data/raw/phits/multilayer/Concrete_45cm_Steel_33cm.out'
# ─────────────────────────────────────────────────────────────────────────────

# ── Layer 1: run k100s2_v1 with TrackNet10 source ────────────────────────────
flux_L1 = predict_flux(model_v1, A_SPEC, MAT1, THICK1)

# Amplitude attenuation factor: how much layer 1 attenuates the beam
# (ratio of integrated flux in 20–150 MeV window vs unshielded reference)
a_factor = flux_L1[BIN_SLICE].sum() / VOID_FLUX[BIN_SLICE].sum()
print(f'Layer-1 amplitude factor (a): {a_factor:.4f}')

# ── Layer 2: run transfer_mlp with TrackNet10 as source ──────────────────────
# Using the original TrackNet10 source (not the normalised CNN L1 output)
# keeps the model in-distribution. The a-factor carries the layer-1 attenuation.
flux_L2_raw = predict_flux(model_transfer, A_SPEC, MAT2, THICK2)

# ── Final flux: scale layer-2 output by layer-1 amplitude ────────────────────
flux_final = flux_L2_raw * a_factor
dose_cnn   = compute_dose(flux_final)

print(f'\n{MAT1} {THICK1}cm  +  {MAT2} {THICK2}cm')
print(f'  CNN dose rate : {dose_cnn:.4e} mrem/hr')

# ── Plot ──────────────────────────────────────────────────────────────────────
color = MAT_COLOR.get(MAT1, '#333')
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(E_ALL, flux_final, color=color, lw=2, label='CNN model (chained)')

if PHITS_ML_FILE:
    phits_flux, _ = load_phits(PHITS_ML_FILE)
    dose_phits    = compute_dose(phits_flux)
    pct_err       = 100 * (dose_cnn - dose_phits) / dose_phits
    ax.semilogy(E_ALL, phits_flux, 'k-', lw=2, label='PHITS')
    ax.set_title(
        f'{MAT1} {THICK1}cm  +  {MAT2} {THICK2}cm   |   dose error {pct_err:+.1f}%',
        fontsize=12, fontweight='bold')
    print(f'  PHITS dose    : {dose_phits:.4e} mrem/hr')
    print(f'  Dose error    : {pct_err:+.1f}%')
else:
    ax.set_title(f'{MAT1} {THICK1}cm  +  {MAT2} {THICK2}cm', fontsize=12, fontweight='bold')

ax.set_xlabel('Neutron energy (MeV)', fontsize=11)
ax.set_ylabel('Flux  [n / cm² / source n]', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.25)
plt.tight_layout()
plt.show()

---
## 6 — Advanced: use any model from experimental/

Load any alternative model and swap it into either inference mode above.

In [ ]:
# ── List available experimental models ────────────────────────────────────────
print('Experimental models:')
for p in sorted(EXPERIMENTAL.iterdir()):
    pkls = list(p.glob('*.pkl'))
    print(f'  {p.name:35s}  →  {pkls[0].name if pkls else "(no pkl)"}')

In [ ]:
# ── Load an experimental model ────────────────────────────────────────────────
# Uncomment and set the subfolder name:

# model_exp = load_pkl(EXPERIMENTAL / 'transfer_conv_head' / 'transfer_model.pkl')
# model_exp = load_pkl(EXPERIMENTAL / 'v_concrete25' / 'model.pkl')

# Then swap into single-layer:
# flux = predict_flux(model_exp, A_SPEC, MATERIAL, THICKNESS_CM)

# Or into multilayer layer-2:
# flux_L2_raw = predict_flux(model_exp, A_SPEC, MAT2, THICK2)
# flux_final  = flux_L2_raw * a_factor   # a_factor from cell 5 above